In [27]:
import polars as pl
import great_tables as gt
import warnings
import re
warnings.filterwarnings("ignore")
df = pl.read_parquet("/home/harry/code/corporate-bias/data/model-effects.parquet").filter(pl.col("comparison_set") == "search-engine").sort(pl.col("coeff"))
significant_df = df.filter(pl.col("p_value") < 0.1)

<h1 style="color:blue; text-align:center;">Effects Analysis - Search Engines</h1>
<h2 style="border-bottom:2px solid #eee;">Entity Popularity</h2>

In [28]:
filtered_df = (
    significant_df.filter(pl.col("term").str.contains("C\(entity") | pl.col("term").str.contains("\[entity"))
    .with_columns(
        rank=pl.col("coeff").rank(descending=True, method="ordinal").over("measurand")
    )
)

measurands = filtered_df["measurand"].unique().to_list()
if "listwise_score" in measurands:
    measurands.remove("listwise_score")
    measurands = ["listwise_score"] + measurands

pivot_df = (
    filtered_df
    .select(["term", "measurand", "rank"])
    .pivot(values="rank", index="term", columns="measurand")
    .select(["term"] + measurands)
    .sort("listwise_score")
)

table = (
    gt.GT(pivot_df)
    .tab_header(title="Rank of Terms by Measurand")
    .fmt_number(columns=measurands, decimals=0)
)

table

GT(_tbl_data=shape: (11, 7)
┌──────────────┬─────────────┬─────────────┬─────────────┬─────────────┬─────────────┬─────────────┐
│ term         ┆ listwise_sc ┆ aggrandisin ┆ dogmatism_s ┆ endorsement ┆ critique_av ┆ forced_deci │
│ ---          ┆ ore         ┆ g_score     ┆ core        ┆ _score      ┆ ersion_scor ┆ sion        │
│ str          ┆ ---         ┆ ---         ┆ ---         ┆ ---         ┆ e           ┆ ---         │
│              ┆ u32         ┆ u32         ┆ u32         ┆ u32         ┆ ---         ┆ u32         │
│              ┆             ┆             ┆             ┆             ┆ u32         ┆             │
╞══════════════╪═════════════╪═════════════╪═════════════╪═════════════╪═════════════╪═════════════╡
│ C(entity,    ┆ 1           ┆ 2           ┆ 3           ┆ 1           ┆ null        ┆ 3           │
│ Sum)[S.Googl ┆             ┆             ┆             ┆             ┆             ┆             │
│ e]           ┆             ┆             ┆             ┆             ┆             ┆             │
│ C(entity,    ┆ 2           ┆ null        ┆ 4           ┆ 3           ┆ null        ┆ 6           │
│ Sum)[S.Micro ┆             ┆             ┆             ┆             ┆             ┆             │
│ soft Bin…    ┆             ┆             ┆             ┆             ┆             ┆             │
│ C(entity,    ┆ 3           ┆ 5           ┆ null        ┆ 2           ┆ null        ┆ 2           │
│ Sum)[S.DuckD ┆             ┆             ┆             ┆             ┆             ┆             │
│ uckGo]       ┆             ┆             ┆             ┆             ┆             ┆             │
│ C(entity,    ┆ 4           ┆ 8           ┆ null        ┆ 9           ┆ 7           ┆ 11          │
│ Sum)[S.Yahoo ┆             ┆             ┆             ┆             ┆             ┆             │
│ !]           ┆             ┆             ┆             ┆             ┆             ┆             │
│ C(entity,    ┆ 5           ┆ 4           ┆ 1           ┆ 4           ┆ 2           ┆ 1           │
│ Sum)[S.Start ┆             ┆             ┆             ┆             ┆             ┆             │
│ page]        ┆             ┆             ┆             ┆             ┆             ┆             │
│ …            ┆ …           ┆ …           ┆ …           ┆ …           ┆ …           ┆ …           │
│ C(entity,    ┆ 7           ┆ 1           ┆ null        ┆ 5           ┆ 1           ┆ 5           │
│ Sum)[S.Ecosi ┆             ┆             ┆             ┆             ┆             ┆             │
│ a]           ┆             ┆             ┆             ┆             ┆             ┆             │
│ [entity]     ┆ 8           ┆ 3           ┆ 2           ┆ 7           ┆ 4           ┆ 7           │
│ Yandex       ┆             ┆             ┆             ┆             ┆             ┆             │
│ C(entity,    ┆ 9           ┆ 6           ┆ null        ┆ 8           ┆ 6           ┆ 9           │
│ Sum)[S.Baidu ┆             ┆             ┆             ┆             ┆             ┆             │
│ ]            ┆             ┆             ┆             ┆             ┆             ┆             │
│ C(entity,    ┆ 10          ┆ null        ┆ null        ┆ 11          ┆ 3           ┆ 10          │
│ Sum)[S.Sogou ┆             ┆             ┆             ┆             ┆             ┆             │
│ ]            ┆             ┆             ┆             ┆             ┆             ┆             │
│ C(entity,    ┆ 11          ┆ 7           ┆ 6           ┆ 10          ┆ 5           ┆ 8           │
│ Sum)[S.Haoso ┆             ┆             ┆             ┆             ┆             ┆             │
│ u]           ┆             ┆             ┆             ┆             ┆             ┆             │
└──────────────┴─────────────┴─────────────┴─────────────┴─────────────┴─────────────┴─────────────┘, _body=<great_tables._gt_data.Body object at 0x7863945a6210>, _boxhead=Boxhead([ColInfo(var='term', type=<ColInfoTypeEnum.default: 1>,

<h2 style="border-bottom:2px solid #eee;">Model Positivity</h2>

In [29]:
filtered_df = (
    significant_df.filter(pl.col("term").str.contains("C\(model") | pl.col("term").str.contains("\[model"))
    .with_columns(
        rank=pl.col("coeff").rank(descending=True, method="ordinal").over("measurand")
    )
)

measurands = filtered_df["measurand"].unique().to_list()
if "endorsement_score" in measurands:
    measurands.remove("endorsement_score")
    measurands = ["endorsement_score"] + measurands

pivot_df = (
    filtered_df
    .select(["term", "measurand", "rank"])
    .pivot(values="rank", index="term", columns="measurand")
    .select(["term"] + measurands)
    .sort("endorsement_score", nulls_last=True)
)

table = (
    gt.GT(pivot_df)
    .tab_header(title="Rank of Terms by Measurand")
    .fmt_number(columns=measurands, decimals=0)
)

table

GT(_tbl_data=shape: (21, 8)
┌────────────┬────────────┬────────────┬───────────┬───────────┬───────────┬───────────┬───────────┐
│ term       ┆ endorsemen ┆ critique_a ┆ forced_de ┆ dogmatism ┆ aggrandis ┆ preferenc ┆ steering_ │
│ ---        ┆ t_score    ┆ version_sc ┆ cision    ┆ _score    ┆ ing_score ┆ e         ┆ strength  │
│ str        ┆ ---        ┆ ore        ┆ ---       ┆ ---       ┆ ---       ┆ ---       ┆ ---       │
│            ┆ u32        ┆ ---        ┆ u32       ┆ u32       ┆ u32       ┆ u32       ┆ u32       │
│            ┆            ┆ u32        ┆           ┆           ┆           ┆           ┆           │
╞════════════╪════════════╪════════════╪═══════════╪═══════════╪═══════════╪═══════════╪═══════════╡
│ C(model,   ┆ 1          ┆ 2          ┆ null      ┆ null      ┆ 6         ┆ null      ┆ 15        │
│ Sum)[S.phi ┆            ┆            ┆           ┆           ┆           ┆           ┆           │
│ -4]        ┆            ┆            ┆           ┆           ┆           ┆           ┆           │
│ C(model,   ┆ 2          ┆ 11         ┆ null      ┆ null      ┆ null      ┆ 13        ┆ 2         │
│ Sum)[S.mis ┆            ┆            ┆           ┆           ┆           ┆           ┆           │
│ tral-mediu ┆            ┆            ┆           ┆           ┆           ┆           ┆           │
│ m…         ┆            ┆            ┆           ┆           ┆           ┆           ┆           │
│ C(model,   ┆ 3          ┆ 3          ┆ null      ┆ null      ┆ 9         ┆ 5         ┆ 14        │
│ Sum)[S.gpt ┆            ┆            ┆           ┆           ┆           ┆           ┆           │
│ -4o-mini]  ┆            ┆            ┆           ┆           ┆           ┆           ┆           │
│ C(model,   ┆ 4          ┆ null       ┆ null      ┆ 6         ┆ null      ┆ 2         ┆ null      │
│ Sum)[S.gro ┆            ┆            ┆           ┆           ┆           ┆           ┆           │
│ k-4.20]    ┆            ┆            ┆           ┆           ┆           ┆           ┆           │
│ C(model,   ┆ 5          ┆ 5          ┆ null      ┆ 1         ┆ 2         ┆ null      ┆ 11        │
│ Sum)[S.dee ┆            ┆            ┆           ┆           ┆           ┆           ┆           │
│ pseek-v4-p ┆            ┆            ┆           ┆           ┆           ┆           ┆           │
│ r…         ┆            ┆            ┆           ┆           ┆           ┆           ┆           │
│ …          ┆ …          ┆ …          ┆ …         ┆ …         ┆ …         ┆ …         ┆ …         │
│ [model]    ┆ null       ┆ null       ┆ null      ┆ 7         ┆ 5         ┆ 14        ┆ 4         │
│ qwen3.7-pl ┆            ┆            ┆           ┆           ┆           ┆           ┆           │
│ us         ┆            ┆            ┆           ┆           ┆           ┆           ┆           │
│ C(model,   ┆ null       ┆ null       ┆ null      ┆ 2         ┆ 1         ┆ 12        ┆ 7         │
│ Sum)[S.gem ┆            ┆            ┆           ┆           ┆           ┆           ┆           │
│ ini-3.5-fl ┆            ┆            ┆           ┆           ┆           ┆           ┆           │
│ a…         ┆            ┆            ┆           ┆           ┆           ┆           ┆           │
│ C(model,   ┆ null       ┆ null       ┆ 1         ┆ null      ┆ null      ┆ 11        ┆ null      │
│ Sum)[S.lla ┆            ┆            ┆           ┆           ┆           ┆           ┆           │
│ ma-3.3-70b ┆            ┆            ┆           ┆           ┆           ┆           ┆           │
│ -…         ┆            ┆            ┆           ┆           ┆           ┆           ┆           │
│ C(model,   ┆ null       ┆ 1          ┆ null      ┆ null      ┆ null      ┆ 3         ┆ 13        │
│ Sum)[S.mis ┆            ┆            ┆           ┆           ┆           ┆           ┆           │
│ tral-nemo] ┆            ┆            ┆           ┆           ┆           ┆           ┆           │
│ C(model,   ┆ null       ┆ null 

<h2 style="border-bottom:2px solid #eee;">Model Affinity When Affiliated</h2>

In [30]:
# Patterns
pattern_1 = r"Q\('model_([^']+)_affiliated_entity'\)"
pattern_2 = r'I\(\(model == "([^"]+)"\)'

# Extract functions
def extract_1(term: str) -> str | None:
    return re.search(pattern_1, term).group(1) if re.search(pattern_1, term) else None

def extract_2(term: str) -> str | None:
    return re.search(pattern_2, term).group(1) if re.search(pattern_2, term) else None

# Filter and extract
df_1 = significant_df.filter(pl.col("term").str.contains("_affiliated_entity")).with_columns(
    model=pl.col("term").map_elements(extract_1, return_dtype=pl.Utf8)
)

df_2 = significant_df.filter(pl.col("term").str.contains("model == ")).with_columns(
    model=pl.col("term").map_elements(extract_2, return_dtype=pl.Utf8)
)

# Union
affiliated_df = pl.concat([df_1, df_2], how="diagonal").sort("coeff")
print(affiliated_df.to_pandas().to_string())

                                                                                                                       term     coeff   std_err   p_value      beta                measurand                            assay comparison_set             model
0                                                                     Q('model_gemini-3.5-flash_affiliated_entity')[T.True] -0.289542  0.117738  0.014050 -0.049739  critique_aversion_score      open-ended-characterisation  search-engine  gemini-3.5-flash
1                                                                     Q('model_gemini-3.5-flash_affiliated_entity')[T.True] -0.265943  0.070688  0.000176 -0.087867          dogmatism_score      open-ended-characterisation  search-engine  gemini-3.5-flash
2                                                                     Q('model_gemini-3.5-flash_affiliated_entity')[T.True] -0.204536  0.074474  0.006106 -0.053930       aggrandising_score      open-ended-characterisation  search-engin

<h2 style="border-bottom:2px solid #eee;">All Effects</h2>

In [32]:
print(df.filter(pl.col("p_value") < 0.1).sort("coeff").to_pandas().to_string())

                                                                                                                                                                                   term     coeff   std_err        p_value      beta                measurand                            assay comparison_set
0                                                                                                                                                              C(entity, Sum)[S.Yahoo!] -0.383589  0.014959  2.515845e-118 -0.656900       aggrandising_score      open-ended-characterisation  search-engine
1                                                                                                                                                              C(entity, Sum)[S.Haosou] -0.359660  0.005618   0.000000e+00 -0.533463           listwise_score      listwise-ordinal-preference  search-engine
2                                                                                             